In [1]:
import shap
import joblib
import pandas as pd
import matplotlib.pyplot as plt

Load the Dataset

In [2]:
df = pd.read_csv("../data/processed_telco_churn.csv")
from sklearn.model_selection import train_test_split
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [3]:
best_lr=joblib.load('../models/customer_churn_model.pkl')

Load the Trained Model

In [4]:
preprocessor = best_lr.named_steps["preprocessor"]

logistic_model = best_lr.named_steps["model"]

In [5]:
X_train_processed = preprocessor.transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [6]:
feature_names = preprocessor.get_feature_names_out()

In [7]:
explainer = shap.LinearExplainer(
    logistic_model,
    X_train_processed
)

Background dataset has 5634 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=5634 when initializing the masker.


In [8]:
shap_values = explainer.shap_values(X_test_processed)

In [9]:
shap.summary_plot(
    shap_values,
    X_test_processed,
    feature_names=feature_names,
    show=False
)

plt.savefig("../plots/shap plots/shap_summary_plot.png",
            dpi=300,
            bbox_inches="tight")

plt.close()

In [ ]:
customer_index=int(input("Enter the customer index:"))

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[customer_index],
        base_values=explainer.expected_value,
        data=X_test_processed[customer_index].toarray().flatten()
        if hasattr(X_test_processed[customer_index], "toarray")
        else X_test_processed[customer_index],
        feature_names=feature_names
    ),
    show=False
)
   
plt.savefig(
    "../plots/shap plots/shap_waterfall_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

